In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import LightSource
import glob

plt.rcParams['figure.figsize'] = (12, 10)
plt.rcParams['font.size'] = 12

## 1. Load Data

We'll load data from both fixed and adaptive path metadynamics simulations.

In [ ]:
# Load the potential energy surface
# Data stored column-major: for each x, all y are listed (gnuplot style)
potential_data = np.loadtxt('Exercise/potential.dat', comments='#')
x_vals = np.unique(potential_data[:, 0])
y_vals = np.unique(potential_data[:, 1])
X, Y = np.meshgrid(x_vals, y_vals)
# Reshape with y varying fastest, then transpose to match meshgrid
Z_potential = potential_data[:, 2].reshape(len(x_vals), len(y_vals)).T

# Load fixed path metadynamics
fixed_colvar = np.loadtxt('Exercise/2_fixedpathmetadynamics/colvar.out')
fixed_time = fixed_colvar[:, 0]
fixed_x = fixed_colvar[:, 1]
fixed_y = fixed_colvar[:, 2]
fixed_s = fixed_colvar[:, 3]
fixed_z = fixed_colvar[:, 4]
fixed_bias = fixed_colvar[:, 5]

# Load adaptive path metadynamics
adaptive_colvar = np.loadtxt('Exercise/3_adaptivepathmetadynamics/colvar.out')
adaptive_time = adaptive_colvar[:, 0]
adaptive_x = adaptive_colvar[:, 1]
adaptive_y = adaptive_colvar[:, 2]
adaptive_s = adaptive_colvar[:, 3]
adaptive_z = adaptive_colvar[:, 4]
adaptive_bias = adaptive_colvar[:, 5]

# Load free energy surfaces
fixed_fes = np.loadtxt('Exercise/2_fixedpathmetadynamics/fes.dat')
adaptive_fes = np.loadtxt('Exercise/3_adaptivepathmetadynamics/fes.dat')

print("Fixed Path Metadynamics:")
print(f"  Trajectory length: {len(fixed_time)} frames")
print(f"  s range: [{fixed_s.min():.2f}, {fixed_s.max():.2f}]")
print(f"  z range: [{fixed_z.min():.3f}, {fixed_z.max():.3f}]")

print("\nAdaptive Path Metadynamics:")
print(f"  Trajectory length: {len(adaptive_time)} frames")
print(f"  s range: [{adaptive_s.min():.2f}, {adaptive_s.max():.2f}]")
print(f"  z range: [{adaptive_z.min():.3f}, {adaptive_z.max():.3f}]")

## 2. Visualize Paths in Real Space

In [ ]:
# Compare trajectories in real space
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Fixed path metadynamics
levels = np.linspace(Z_potential.min(), 0, 30)
contour1 = ax1.contourf(X, Y, Z_potential, levels=levels, cmap='viridis', alpha=0.8)
ax1.contour(X, Y, Z_potential, levels=levels, colors='white', alpha=0.3, linewidths=0.5)
scatter1 = ax1.scatter(fixed_x, fixed_y, c=fixed_time, s=2, cmap='hot', 
                      alpha=0.6, zorder=10)
ax1.plot(fixed_x[0], fixed_y[0], 'go', markersize=15, label='Start', zorder=20)
ax1.plot(fixed_x[-1], fixed_y[-1], 'r*', markersize=20, label='End', zorder=20)
plt.colorbar(contour1, ax=ax1, label='Potential Energy (K)', pad=0.12)
plt.colorbar(scatter1, ax=ax1, label='Time (steps)', fraction=0.046, pad=0.04)
ax1.set_xlabel('cv.x')
ax1.set_ylabel('cv.y')
ax1.set_title('Fixed Path Metadynamics Trajectory')
ax1.set_xlim([-1.5, 1.5])
ax1.set_ylim([-0.5, 2.5])
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Adaptive path metadynamics
contour2 = ax2.contourf(X, Y, Z_potential, levels=levels, cmap='viridis', alpha=0.8)
ax2.contour(X, Y, Z_potential, levels=levels, colors='white', alpha=0.3, linewidths=0.5)
scatter2 = ax2.scatter(adaptive_x, adaptive_y, c=adaptive_time, s=2, cmap='hot', 
                      alpha=0.6, zorder=10)
ax2.plot(adaptive_x[0], adaptive_y[0], 'go', markersize=15, label='Start', zorder=20)
ax2.plot(adaptive_x[-1], adaptive_y[-1], 'r*', markersize=20, label='End', zorder=20)
plt.colorbar(contour2, ax=ax2, label='Potential Energy (K)', pad=0.12)
plt.colorbar(scatter2, ax=ax2, label='Time (steps)', fraction=0.046, pad=0.04)
ax2.set_xlabel('cv.x')
ax2.set_ylabel('cv.y')
ax2.set_title('Adaptive Path Metadynamics Trajectory')
ax2.set_xlim([-1.5, 1.5])
ax2.set_ylim([-0.5, 2.5])
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Path Collective Variables

The key innovation of path metadynamics is the use of path collective variables (s, z) instead of the original coordinates.

In [ ]:
# Plot evolution of path CVs over time
fig, axes = plt.subplots(3, 2, figsize=(16, 12), sharex='col')

# Fixed path
axes[0, 0].plot(fixed_time, fixed_s, linewidth=0.8)
axes[0, 0].set_ylabel('s (progress)')
axes[0, 0].set_title('Fixed Path Metadynamics')
axes[0, 0].grid(True, alpha=0.3)

axes[1, 0].plot(fixed_time, fixed_z, linewidth=0.8, color='orange')
axes[1, 0].set_ylabel('z (distance from path)')
axes[1, 0].grid(True, alpha=0.3)

axes[2, 0].plot(fixed_time, fixed_bias, linewidth=0.8, color='red')
axes[2, 0].set_ylabel('Bias Potential (K)')
axes[2, 0].set_xlabel('Time (steps)')
axes[2, 0].grid(True, alpha=0.3)

# Adaptive path
axes[0, 1].plot(adaptive_time, adaptive_s, linewidth=0.8)
axes[0, 1].set_ylabel('s (progress)')
axes[0, 1].set_title('Adaptive Path Metadynamics')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 1].plot(adaptive_time, adaptive_z, linewidth=0.8, color='orange')
axes[1, 1].set_ylabel('z (distance from path)')
axes[1, 1].grid(True, alpha=0.3)

axes[2, 1].plot(adaptive_time, adaptive_bias, linewidth=0.8, color='red')
axes[2, 1].set_ylabel('Bias Potential (K)')
axes[2, 1].set_xlabel('Time (steps)')
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 2D visualization in path CV space
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Fixed path
scatter1 = ax1.scatter(fixed_s, fixed_z, c=fixed_time, s=3, 
                      cmap='viridis', alpha=0.6)
ax1.axhline(y=0, color='r', linestyle='--', alpha=0.5, linewidth=2, 
           label='Path (z=0)')
plt.colorbar(scatter1, ax=ax1, label='Time (steps)')
ax1.set_xlabel('s (progress along path)')
ax1.set_ylabel('z (distance from path)')
ax1.set_title('Fixed Path: Sampling in Path CV Space')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Adaptive path
scatter2 = ax2.scatter(adaptive_s, adaptive_z, c=adaptive_time, s=3, 
                      cmap='viridis', alpha=0.6)
ax2.axhline(y=0, color='r', linestyle='--', alpha=0.5, linewidth=2, 
           label='Path (z=0)')
plt.colorbar(scatter2, ax=ax2, label='Time (steps)')
ax2.set_xlabel('s (progress along path)')
ax2.set_ylabel('z (distance from path)')
ax2.set_title('Adaptive Path: Sampling in Path CV Space')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Free Energy Profiles Along the Path

The main output of path metadynamics is the free energy profile F(s) along the reaction coordinate s.

In [ ]:
# Plot and compare free energy profiles
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Fixed path FES
fixed_s_grid = fixed_fes[:, 0]
fixed_F = fixed_fes[:, 1]
fixed_F_shifted = fixed_F - fixed_F.min()

ax1.plot(fixed_s_grid, fixed_F_shifted, linewidth=2.5, color='blue', 
        label='Fixed Path')
ax1.fill_between(fixed_s_grid, 0, fixed_F_shifted, alpha=0.3, color='blue')
ax1.set_ylabel('Free Energy (K)')
ax1.set_title('Free Energy Profile: Fixed Path Metadynamics')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, max(fixed_F_shifted.max(), 30)])

# Adaptive path FES
adaptive_s_grid = adaptive_fes[:, 0]
adaptive_F = adaptive_fes[:, 1]
adaptive_F_shifted = adaptive_F - adaptive_F.min()

ax2.plot(adaptive_s_grid, adaptive_F_shifted, linewidth=2.5, color='green', 
        label='Adaptive Path')
ax2.fill_between(adaptive_s_grid, 0, adaptive_F_shifted, alpha=0.3, color='green')
ax2.set_xlabel('s (progress along path)')
ax2.set_ylabel('Free Energy (K)')
ax2.set_title('Free Energy Profile: Adaptive Path Metadynamics')
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, max(adaptive_F_shifted.max(), 30)])

plt.tight_layout()
plt.show()

# Find and report barriers
print("\nFree Energy Barriers:")
print("="*50)
print(f"Fixed Path:")
print(f"  Maximum barrier: {fixed_F_shifted.max():.2f} K")
print(f"  Location (s): {fixed_s_grid[np.argmax(fixed_F_shifted)]:.3f}")
print(f"\nAdaptive Path:")
print(f"  Maximum barrier: {adaptive_F_shifted.max():.2f} K")
print(f"  Location (s): {adaptive_s_grid[np.argmax(adaptive_F_shifted)]:.3f}")

In [ ]:
# Direct comparison
fig, ax = plt.subplots(figsize=(14, 8))

ax.plot(fixed_s_grid, fixed_F_shifted, linewidth=2.5, color='blue', 
       label='Fixed Path', marker='o', markersize=4, alpha=0.7)
ax.plot(adaptive_s_grid, adaptive_F_shifted, linewidth=2.5, color='green', 
       label='Adaptive Path', marker='s', markersize=4, alpha=0.7)

ax.set_xlabel('s (progress along path)', fontsize=14)
ax.set_ylabel('Free Energy (K)', fontsize=14)
ax.set_title('Comparison of Free Energy Profiles', fontsize=16, fontweight='bold')
ax.legend(fontsize=12, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Path Evolution Analysis

For adaptive path metadynamics, the path itself evolves to find the minimum free energy path. Let's analyze this evolution.

In [ ]:
import os

path_file = 'Exercise/3_adaptivepathmetadynamics/path.out'

if os.path.exists(path_file):
    # Parse path file (blocks with step/time header + node list)
    with open(path_file, 'r') as f:
        lines = f.readlines()
    
    path_snapshots = []
    current_snapshot = []
    block_time = None  # time from the block header (not node index)
    reading_nodes = False

    for line in lines:
        if line.startswith('#! FIELDS') or line.startswith('#! SET'):
            # End any current snapshot when a new header begins
            if current_snapshot and block_time is not None:
                path_snapshots.append((block_time, np.array(current_snapshot)))
                current_snapshot = []
            reading_nodes = False
            continue

        if line.strip() == '':
            # Blank line ends a block
            if current_snapshot and block_time is not None:
                path_snapshots.append((block_time, np.array(current_snapshot)))
                current_snapshot = []
            reading_nodes = False
            block_time = None
            continue

        parts = line.split()

        # Block header with Step Time nCV nNodes (4 numbers)
        if len(parts) == 4 and not reading_nodes:
            try:
                block_time = float(parts[1])
                reading_nodes = False
                continue
            except ValueError:
                pass

        # After the node header, we read node lines: node, x, y, ...
        if len(parts) >= 3:
            # If we have not yet seen the node header, assume these are nodes
            reading_nodes = True
            x, y = float(parts[1]), float(parts[2])
            current_snapshot.append([x, y])

    # Flush last snapshot
    if current_snapshot and block_time is not None:
        path_snapshots.append((block_time, np.array(current_snapshot)))
    
    print(f"Loaded {len(path_snapshots)} path snapshots")
    
    # Plot path evolution
    if len(path_snapshots) > 0:
        fig, ax = plt.subplots(figsize=(9, 7))
        
        # Plot potential
        levels = np.linspace(Z_potential.min(), 0, 30)
        contour = ax.contourf(X, Y, Z_potential, levels=levels, cmap='viridis', alpha=0.7)
        ax.contour(X, Y, Z_potential, levels=levels, colors='white', alpha=0.3, linewidths=0.5)
        
        # Plot path evolution
        n_paths_to_show = min(10, len(path_snapshots))
        indices = np.linspace(0, len(path_snapshots)-1, n_paths_to_show, dtype=int)
        
        for i, idx in enumerate(indices):
            time, path = path_snapshots[idx]
            color = plt.cm.plasma(i / max(1, n_paths_to_show - 1))
            ax.plot(path[:, 0], path[:, 1], '-o', color=color,
                    linewidth=2, markersize=4, alpha=0.8,
                    label=f't={time:.0f}')
        
        plt.colorbar(contour, ax=ax, label='Potential Energy (K)')
        ax.set_xlabel('cv.x')
        ax.set_ylabel('cv.y')
        ax.set_title('Evolution of Adaptive Path', fontsize=16, fontweight='bold')
        ax.set_xlim([-1.5, 1.5])
        ax.set_ylim([-0.5, 2.5])
        ax.legend(loc='upper right', fontsize=10, ncol=2)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print(f"Path file not found: {path_file}")
    print("The path evolution cannot be visualized.")

## 6. Sampling Efficiency Comparison

In [ ]:
# Compare sampling along s coordinate
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Histograms of s
axes[0, 0].hist(fixed_s, bins=50, alpha=0.7, color='blue', 
               edgecolor='black', linewidth=0.5)
axes[0, 0].set_xlabel('s (progress)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Fixed Path: Distribution of s')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(adaptive_s, bins=50, alpha=0.7, color='green', 
               edgecolor='black', linewidth=0.5)
axes[0, 1].set_xlabel('s (progress)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Adaptive Path: Distribution of s')
axes[0, 1].grid(True, alpha=0.3)

# Histograms of z
axes[1, 0].hist(fixed_z, bins=50, alpha=0.7, color='blue', 
               edgecolor='black', linewidth=0.5)
axes[1, 0].set_xlabel('z (distance from path)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Fixed Path: Distribution of z')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(adaptive_z, bins=50, alpha=0.7, color='green', 
               edgecolor='black', linewidth=0.5)
axes[1, 1].set_xlabel('z (distance from path)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Adaptive Path: Distribution of z')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print("\nSampling Statistics:")
print("="*50)
print("\nFixed Path:")
print(f"  <z> = {np.mean(fixed_z):.4f} ± {np.std(fixed_z):.4f}")
print(f"  max|z| = {np.max(np.abs(fixed_z)):.4f}")
print("\nAdaptive Path:")
print(f"  <z> = {np.mean(adaptive_z):.4f} ± {np.std(adaptive_z):.4f}")
print(f"  max|z| = {np.max(np.abs(adaptive_z)):.4f}")

## Conclusions

From these path metadynamics simulations, we have learned:

### 1. Fixed Path Metadynamics

**Advantages:**
- Focuses sampling along a predefined pathway
- More efficient than standard metadynamics for studying specific transitions
- Provides direct free energy profile along the path

**Limitations:**
- Requires prior knowledge of a reasonable path
- The initial path guess affects the results
- May miss the actual minimum free energy path

### 2. Adaptive Path Metadynamics

**Advantages:**
- Path evolves to find the minimum free energy path (MFEP)
- Does not require detailed prior knowledge
- Automatically optimizes the transition pathway
- Generally gives lower barrier heights (closer to true MFEP)

**Key Features:**
- The path is updated periodically based on sampling
- Nodes redistribute to maintain equal spacing along the path
- Can handle complex, multidimensional transition pathways

### 3. Comparison with Standard Metadynamics

Path metadynamics offers several advantages:
1. **Efficiency**: Focuses on relevant transition region
2. **Lower dimensionality**: Uses 1D coordinate (s) instead of full CV space
3. **Physical insight**: Directly provides reaction coordinate and free energy barrier
4. **Scalability**: Better for systems with many degrees of freedom

### 4. Practical Considerations

**Parameter choices:**
- **Number of path nodes**: Balance between resolution and computational cost
- **λ parameter**: Controls smoothness of path collective variables
- **Update frequency**: How often the path is re-optimized (adaptive)
- **Fixed nodes**: Anchor points to maintain start/end states

**Applications:**
- Protein conformational changes
- Chemical reactions in solution
- Phase transitions
- Ligand binding/unbinding pathways

### Summary

Path metadynamics represents a powerful extension of standard metadynamics, particularly suited for:
- Systems where the reaction coordinate is not obvious
- Complex transitions involving many degrees of freedom
- Cases where computational efficiency is important
- Studies requiring precise barrier heights and transition state structures

The adaptive variant is generally preferred as it discovers the optimal pathway automatically, though it requires more careful parameter tuning and longer simulations to converge.